# 🌾 Soil Health Monitoring with UAV Imaging using Machine Learning for Sustainable Farming
### **Pan-India Precision Agriculture Framework (7 Major Indian Soil Types)**
- **Benchmark Dataset:** Elsevier EAAI Research Dataset (6,284 Authentic Images Across India)
- **Principal Investigator:** Dr. Ritu Garg (Associate Professor, Computer Engineering Dept, NIT Kurukshetra)
- **Co-PI:** Dr. Amandeep Kaur (Assistant Professor, Computer Engineering Dept, NIT Kurukshetra)
- **Theme:** Precision Agriculture, Unmanned Aerial Vehicles (UAVs), Deep Learning & Computer Vision

---
## 📌 Project Objectives:
1. **UAV Aerial Image Acquisition & Preprocessing:** Bilateral filtering, CLAHE contrast enhancement, LAB/HSV color space transforms, and bare-soil vegetation masking.
2. **Multi-Domain Feature Extraction:** 60 handcrafted features including Gray-Level Co-occurrence Matrix (GLCM Haralick texture), Local Binary Patterns (LBP), and color moments.
3. **Proposed Novel Model (`UAV-SoilHealthNet`):** Dual-Branch Cross-Attention Multi-Task Network outperforming the Elsevier base paper across 7 Indian Soil Types.
4. **AI Explainability:** Grad-CAM attention heatmaps validating learned soil surface cues.
5. **Digital GIS Soil Health Mapping:** Color-coded field spatial zoning (Healthy, Moderate, Degraded zones) and continuous Soil Health Index (SHI: 0-100).
6. **Scientifically Valid Farmer Advisory:** ICAR & State Agricultural Universities aligned agronomic prescriptions across all Indian agro-climatic zones without unscientific RGB N-P-K claims.

In [ ]:
import os
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

# Ensure project root is imported
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from src.config import SOIL_TYPES, SOIL_HEALTH_STATUSES, RAW_DATA_DIR, SAMPLES_DIR, REPORTS_DIR, MAPS_DIR
from src.preprocessing.preprocessor import UAVImagePreprocessor
from src.preprocessing.feature_extractor import SoilFeatureExtractor
from src.models.advanced_model import UAVSoilHealthNet
from src.models.explainability import SoilGradCAM
from src.advisory.farmer_advisory import FarmerAdvisoryEngine

print("✓ Core modules successfully imported.")
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"✓ Hardware Device: {device}")

## 🧪 1. Drone Image Preprocessing & Contrast Enhancement (CLAHE)

In [ ]:
preproc = UAVImagePreprocessor()

# Load a real Indian soil sample from data/raw/Alluvial_Soil
sample_files = sorted(list((RAW_DATA_DIR / 'Alluvial_Soil').glob('*.*')))
raw_bgr = cv2.imread(str(sample_files[0]))
sample_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)

denoised = preproc.denoise_image(sample_rgb, method='bilateral')
enhanced = preproc.enhance_contrast_clahe(denoised)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), dpi=120)
axes[0].imshow(sample_rgb); axes[0].set_title("Raw Field RGB Image"); axes[0].axis('off')
axes[1].imshow(denoised); axes[1].set_title("Bilateral Filtered (Denoised)"); axes[1].axis('off')
axes[2].imshow(enhanced); axes[2].set_title("LAB CLAHE Contrast Enhanced"); axes[2].axis('off')
plt.tight_layout()
plt.show()

## 🔬 2. Handcrafted Feature Extraction (GLCM Texture + LBP + Color Moments)

In [ ]:
feat_extractor = SoilFeatureExtractor()
features = feat_extractor.extract_all_features(enhanced)

print(f"Extracted Feature Vector Length: {len(features)}")
print(f"Color Moments (RGB, HSV, LAB) : {features[:27].shape[0]} features")
print(f"GLCM Haralick Texture Descriptors: {features[27:39].shape[0]} features (Contrast, Energy, Homogeneity, ASM)")
print(f"Local Binary Patterns (LBP)   : {features[39:55].shape[0]} features")
print(f"Physical Optical Proxies      : {features[55:].shape[0]} features (Organic proxy, roughness, moisture)")

## 🚀 3. Proposed Novel Model Inference & Grad-CAM Explainability

In [ ]:
model = UAVSoilHealthNet(num_soil_types=7, num_health_states=3, num_handcrafted_features=60, pretrained_backbone=False).to(device)
ckpt_path = project_root / 'outputs' / 'models' / 'uav_soilhealthnet_best.pt'
if ckpt_path.exists():
    model.load_state_dict(torch.load(str(ckpt_path), map_location=device))
model.eval()

t_img = torch.from_numpy(enhanced.transpose(2, 0, 1)).float().unsqueeze(0).to(device) / 255.0
t_hc = torch.tensor(features, dtype=torch.float32).unsqueeze(0).to(device)

with torch.no_grad():
    out = model(t_img, t_hc)
    s_idx = int(torch.argmax(out['soil_type_logits'], dim=1).item())
    h_idx = int(torch.argmax(out['health_status_logits'], dim=1).item())
    shi = float(out['soil_health_index'].item())

print(f"🎯 Predicted Soil Type   : {SOIL_TYPES[s_idx]}")
print(f"🌱 Predicted Health Status : {SOIL_HEALTH_STATUSES[h_idx]}")
print(f"📈 Soil Health Index (SHI): {shi:.1f} / 100")

# Grad-CAM Overlay
grad_cam = SoilGradCAM(model=model, target_layer=model.deep_features[-1])
cam = grad_cam.generate_cam(t_img, handcrafted_features=t_hc)
blended, _ = SoilGradCAM.overlay_heatmap(enhanced, cam)

plt.figure(figsize=(6, 6), dpi=120)
plt.imshow(blended)
plt.title(f"Grad-CAM Visual Attention Overlay\n({SOIL_TYPES[s_idx]} - {SOIL_HEALTH_STATUSES[h_idx]})")
plt.axis('off')
plt.show()

## 🗺️ 4. Digital Soil Health GIS Spatial Field Map

In [ ]:
map_path = project_root / 'outputs' / 'maps' / 'Indian_Agricultural_Field_Zone_A_soil_health_map.png'
if map_path.exists():
    img_map = cv2.imread(str(map_path))
    plt.figure(figsize=(16, 6), dpi=150)
    plt.imshow(cv2.cvtColor(img_map, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title("UAV Aerial Field Zoning Map (Pan-India Agriculture)", fontsize=14, fontweight='bold')
    plt.show()
else:
    print("Map image will be generated when running run_pipeline.py.")

## 🌾 5. Actionable ICAR Pan-India Farmer Prescription

In [ ]:
advisory_engine = FarmerAdvisoryEngine()
adv = advisory_engine.generate_advisory(SOIL_TYPES[s_idx], SOIL_HEALTH_STATUSES[h_idx], shi, "Indo-Gangetic Basin Farmland")
print(advisory_engine.format_farmer_card_text(adv))